# Milestone 3: Text Vectorization & Embeddings

This notebook covers two text-based milestones in deep learning:
1. **Sentiment Classification**: Predicting review sentiment (Positive/Negative) from review text. We compare a classical **Bag-of-Words/TF-IDF baseline** against a **Keras Learned Embeddings model**.
2. **Semantic Product Search**: Building an embedding-based "similar products" search engine using cosine similarity over product description embeddings.

### Roadmap
1. **Environment Setup & Colab Compatibility**
2. **Sentiment Classification (Part A)**:
   - Data Preparation: Extracting text features and labeling binary sentiment
   - TF-IDF + Logistic Regression Baseline
   - Keras Tokenizer & Word Embedding Model
   - Model Comparison (Accuracy, F1-score)
3. **Semantic Product Search (Part B)**:
   - Extracting product description embeddings using average word embeddings
   - Implementing a cosine similarity search engine to retrieve similar products given a search query

## 1. Setup, Environment Detection & Imports

In [ ]:
import sys
import os
from pathlib import Path

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab. Installing compatible datasets library (<3.0.0) and other packages...")
    !pip install "datasets>=2.16.0,<3.0.0" transformers diffusers accelerate gradio -q 2>/dev/null

    # Create folders
    os.makedirs('src', exist_ok=True)
    os.makedirs('data/processed', exist_ok=True)
    os.makedirs('data/plots', exist_ok=True)

    # Write data.py directly to Colab disk for imports
    data_py_content = """import os
import json
import pandas as pd
import numpy as np
import requests
from pathlib import Path
from sklearn.model_selection import train_test_split
from datasets import load_dataset
import io
from PIL import Image
from tqdm import tqdm

def load_amazon_data(category="All_Beauty"):
    print(f"Loading reviews for {category}...")
    reviews_dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", f"raw_review_{category}", trust_remote_code=True)
    reviews_df = pd.DataFrame(reviews_dataset['full'])

    print(f"Loading metadata for {category}...")
    meta_dataset = load_dataset("McAuley-Lab/Amazon-Reviews-2023", f"raw_meta_{category}", split="full", trust_remote_code=True)
    meta_df = pd.DataFrame(meta_dataset)
    return reviews_df, meta_df

def clean_data(reviews_df, meta_df):
    print("Cleaning metadata...")
    def clean_price(x):
        if pd.isna(x):
            return np.nan
        if isinstance(x, (int, float)):
            return float(x)
        x_str = str(x).replace('$', '').replace(',', '').strip()
        try:
            return float(x_str)
        except ValueError:
            return np.nan
    meta_df['price_cleaned'] = meta_df['price'].apply(clean_price)
    # Process images: extract the first image URL if it is a list of dicts/strings
    def extract_image_url(images_val):
        if not images_val or pd.isna(images_val):
            return None

        def first_string(val):
            if isinstance(val, list):
                if len(val) > 0:
                    first = val[0]
                    if isinstance(first, str):
                        return first
                    elif isinstance(first, dict) and 'large' in first:
                        return first['large']
            elif isinstance(val, str):
                return val
            return None

        if isinstance(images_val, dict):
            for key in ['large', 'hi_res', 'thumb']:
                if key in images_val:
                    res = first_string(images_val[key])
                    if res:
                        return res
            return None

        if isinstance(images_val, (list, np.ndarray)):
            return first_string(list(images_val))

        return None
    meta_df['image_url'] = meta_df['images'].apply(extract_image_url)

    def clean_description(desc):
        if isinstance(desc, list):
            return " ".join([str(d) for d in desc])
        if pd.isna(desc):
            return ""
        return str(desc)
    meta_df['description_cleaned'] = meta_df['description'].apply(clean_description)

    print("Cleaning reviews...")
    reviews_df['rating'] = pd.to_numeric(reviews_df['rating'], errors='coerce')
    vote_col = 'helpful_vote' if 'helpful_vote' in reviews_df.columns else 'helpful_votes'
    if vote_col in reviews_df.columns:
        reviews_df['helpful_votes'] = pd.to_numeric(reviews_df[vote_col], errors='coerce').fillna(0).astype(int)
    else:
        reviews_df['helpful_votes'] = 0
    reviews_df['text_cleaned'] = reviews_df['text'].fillna("")
    reviews_df['title_cleaned'] = reviews_df['title'].fillna("")
    return reviews_df, meta_df

def split_by_product(reviews_df, meta_df, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_seed=42):
    print("Splitting data by product...")
    prod_col = 'parent_asin' if ('parent_asin' in meta_df.columns and 'parent_asin' in reviews_df.columns) else 'asin'
    unique_products = list(set(meta_df[prod_col].unique()) & set(reviews_df[prod_col].unique()))

    train_prods, test_val_prods = train_test_split(unique_products, train_size=train_ratio, random_state=random_seed)
    val_relative_ratio = val_ratio / (val_ratio + test_ratio)
    val_prods, test_prods = train_test_split(test_val_prods, train_size=val_relative_ratio, random_state=random_seed)

    train_prods_set = set(train_prods)
    val_prods_set = set(val_prods)
    test_prods_set = set(test_prods)

    meta_train = meta_df[meta_df[prod_col].isin(train_prods_set)]
    meta_val = meta_df[meta_df[prod_col].isin(val_prods_set)]
    meta_test = meta_df[meta_df[prod_col].isin(test_prods_set)]

    reviews_train = reviews_df[reviews_df[prod_col].isin(train_prods_set)]
    reviews_val = reviews_df[reviews_df[prod_col].isin(val_prods_set)]
    reviews_test = reviews_df[reviews_df[prod_col].isin(test_prods_set)]
    return (reviews_train, meta_train), (reviews_val, meta_val), (reviews_test, meta_test)
"""
    with open('src/data.py', 'w', encoding='utf-8') as f:
        f.write(data_py_content)
    sys.path.append(os.path.abspath('src'))
    data_dir_path = 'data/processed'
    plots_dir_path = 'data/plots'
else:
    sys.path.append(os.path.abspath('../src'))
    data_dir_path = '../data/processed'
    plots_dir_path = '../data/plots'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Sklearn tools
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix
from sklearn.metrics.pairwise import cosine_similarity

# TensorFlow & Keras
import tensorflow as tf
from tensorflow import keras
from keras import layers, models, callbacks

# Set styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
print(f"TensorFlow Version: {tf.__version__}")

Running in Google Colab. Installing compatible datasets library (<3.0.0) and other packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 9.8 MB/s eta 0:00:00
TensorFlow Version: 2.20.0


## 2. Load Processed Parquet Data

In [ ]:
from data import load_amazon_data, clean_data, split_by_product

# Check if processed splits exist
train_meta_path = Path(data_dir_path) / "meta_train.parquet"

if not train_meta_path.exists():
    print("Processed dataset splits not found. Loading and generating splits from scratch...")
    reviews_df, meta_df = load_amazon_data("All_Beauty")
    reviews_df, meta_df = clean_data(reviews_df, meta_df)

    splits = split_by_product(reviews_df, meta_df)
    (reviews_train, meta_train), (reviews_val, meta_val), (reviews_test, meta_test) = splits

    # Save splits
    os.makedirs(data_dir_path, exist_ok=True)
    reviews_train.to_parquet(Path(data_dir_path) / "reviews_train.parquet", index=False)
    reviews_val.to_parquet(Path(data_dir_path) / "reviews_val.parquet", index=False)
    reviews_test.to_parquet(Path(data_dir_path) / "reviews_test.parquet", index=False)

    meta_train.to_parquet(Path(data_dir_path) / "meta_train.parquet", index=False)
    meta_val.to_parquet(Path(data_dir_path) / "meta_val.parquet", index=False)
    meta_test.to_parquet(Path(data_dir_path) / "meta_test.parquet", index=False)
else:
    print("Loading processed splits from disk...")
    reviews_train = pd.read_parquet(Path(data_dir_path) / "reviews_train.parquet")
    reviews_val = pd.read_parquet(Path(data_dir_path) / "reviews_val.parquet")
    reviews_test = pd.read_parquet(Path(data_dir_path) / "reviews_test.parquet")

    meta_train = pd.read_parquet(Path(data_dir_path) / "meta_train.parquet")
    meta_val = pd.read_parquet(Path(data_dir_path) / "meta_val.parquet")
    meta_test = pd.read_parquet(Path(data_dir_path) / "meta_test.parquet")

print(f"Loaded {len(reviews_train)} train reviews, {len(reviews_test)} test reviews.")

Processed dataset splits not found. Loading and generating splits from scratch...
Loading reviews for All_Beauty...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating full split: 0 examples [00:00, ? examples/s]

Loading metadata for All_Beauty...


Generating full split:   0%|          | 0/112590 [00:00<?, ? examples/s]

Cleaning metadata...
Cleaning reviews...
Splitting data by product...
Loaded 493730 train reviews, 106623 test reviews.


## Part A: Sentiment Classification

### 1. Data Preparation
We map raw product ratings (1 to 5) to binary labels:
- **Positive (1)**: Ratings of 4 or 5 stars.
- **Negative (0)**: Ratings of 1 or 2 stars.
- Neutral 3-star reviews are excluded to ensure clean classification boundaries.

In [ ]:
def prepare_sentiment_df(df):
    # Exclude 3-star reviews
    df_filtered = df[df['rating'] != 3].copy()
    # Map 4,5 -> 1 (Positive) and 1,2 -> 0 (Negative)
    df_filtered['sentiment'] = df_filtered['rating'].apply(lambda x: 1 if x > 3 else 0)
    # Drop rows with empty text
    df_filtered = df_filtered[df_filtered['text_cleaned'] != ""]
    return df_filtered[['text_cleaned', 'sentiment']]

sent_train = prepare_sentiment_df(reviews_train)
sent_val = prepare_sentiment_df(reviews_val)
sent_test = prepare_sentiment_df(reviews_test)

print(f"Sentiment Dataset Sizes: Train={len(sent_train)}, Val={len(sent_val)}, Test={len(sent_test)}")
print("Train class distribution:")
print(sent_train['sentiment'].value_counts(normalize=True))

Sentiment Dataset Sizes: Train=454084, Val=93016, Test=98018
Train class distribution:
sentiment
1    0.77539
0    0.22461
Name: proportion, dtype: float64


### 2. Baseline Model: TF-IDF + Logistic Regression
We construct TF-IDF word vectors (limiting vocabulary to 5,000 words) and train a `LogisticRegression` classifier.

In [ ]:
print("Vectorizing text using TF-IDF...")
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(sent_train['text_cleaned'])
X_test_tfidf = vectorizer.transform(sent_test['text_cleaned'])

y_train_sent = sent_train['sentiment'].values
y_test_sent = sent_test['sentiment'].values

print("Training Logistic Regression baseline...")
baseline_lr = LogisticRegression(class_weight='balanced', max_iter=500, random_state=42)
baseline_lr.fit(X_train_tfidf, y_train_sent)

# Evaluate
lr_train_preds = baseline_lr.predict(X_train_tfidf)
lr_test_preds = baseline_lr.predict(X_test_tfidf)

lr_test_acc = accuracy_score(y_test_sent, lr_test_preds)
lr_test_macro_f1 = f1_score(y_test_sent, lr_test_preds, average='macro')

print(f"TF-IDF Baseline Test Accuracy: {lr_test_acc:.4f}")
print(f"TF-IDF Baseline Test Macro F1: {lr_test_macro_f1:.4f}\n")
print(classification_report(y_test_sent, lr_test_preds, target_names=['Negative', 'Positive']))

Vectorizing text using TF-IDF...
Training Logistic Regression baseline...
TF-IDF Baseline Test Accuracy: 0.8864
TF-IDF Baseline Test Macro F1: 0.8523

              precision    recall  f1-score   support

    Negative       0.69      0.90      0.78     22163
    Positive       0.97      0.88      0.92     75855

    accuracy                           0.89     98018
   macro avg       0.83      0.89      0.85     98018
weighted avg       0.90      0.89      0.89     98018



### 3. Deep Learning Model: Keras Learned Embeddings
We train a neural network that maps reviews to dense word embeddings learned from scratch during training. We use a Keras `TextVectorization` layer to tokenize and pad sequences.

In [ ]:
VOCAB_SIZE = 10000
MAX_LEN = 150
EMBEDDING_DIM = 64

# Define text vectorizer
vectorize_layer = layers.TextVectorization(
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)

# Adapt vectorizer over training texts
vectorize_layer.adapt(sent_train['text_cleaned'].values)

def build_embeddings_model():
    model = keras.Sequential([
        layers.Input(shape=(1,), dtype=tf.string),
        vectorize_layer,
        layers.Embedding(input_dim=VOCAB_SIZE, output_dim=EMBEDDING_DIM),
        layers.GlobalAveragePooling1D(),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

embed_model = build_embeddings_model()
embed_model.summary()

# Compile
embed_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

# Train
print("Training Keras Learned Embeddings model...")
history = embed_model.fit(
    sent_train['text_cleaned'].values, y_train_sent,
    validation_data=(sent_val['text_cleaned'].values, sent_val['sentiment'].values),
    epochs=15,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 150)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 150, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 642,113 (2.45 MB)

 Trainable params: 642,113 (2.45 MB)

 Non-trainable params: 0 (0.00 B)

Training Keras Learned Embeddings model...
Epoch 1/15
7096/7096 ━━━━━━━━━━━━━━━━━━━━ 74s 10ms/step - accuracy: 0.8861 - loss: 0.2790 - val_accuracy: 0.8974 - val_loss: 0.2524
Epoch 2/15
7096/7096 ━━━━━━━━━━━━━━━━━━━━ 77s 11ms/step - accuracy: 0.9180 - loss: 0.2070 - val_accuracy: 0.8797 - val_loss: 0.2643
Epoch 3/15
7096/7096 ━━━━━━━━━━━━━━━━━━━━ 70s 10ms/step - accuracy: 0.9223 - loss: 0.1965 - val_accuracy: 0.9132 - val_loss: 0.2119
Epoch 4/15
7096/7096 ━━━━━━━━━━━━━━━━━━━━ 81s 10ms/step - accuracy: 0.9251 - loss: 0.1898 - val_accuracy: 0.9296 - val_loss: 0.1770
Epoch 5/15
7096/7096 ━━━━━━━━━━━━━━━━━━━━ 72s 10ms/step - accuracy: 0.9263 - loss: 0.1865 - val_accuracy: 0.9167 - val_loss: 0.1997
Epoch 6/15
7096/7096 ━━━━━━━━━━━━━━━━━━━━ 78s 10ms/step - accuracy: 0.9277 - loss: 0.1833 - val_accuracy: 0.9210 - val_loss: 0.2093
Epoch 7/15
7096/7096 ━━━━━━━━━━━━━━━━━━━━ 81s 9ms/step - accuracy: 0.9298 - loss: 0.1788 - val_accuracy: 0.9323 - val_loss: 0.1727
Epoch 8/15
7096/7096 ━━━━━━━━━━━━━

### 4. Evaluate Embedding Model & Compare

In [ ]:
# Predict on test set
embed_probs = embed_model.predict(sent_test['text_cleaned'].values).squeeze()
embed_preds = (embed_probs >= 0.5).astype(int)

embed_test_acc = accuracy_score(y_test_sent, embed_preds)
embed_test_macro_f1 = f1_score(y_test_sent, embed_preds, average='macro')

print(f"Embedding Model Test Accuracy: {embed_test_acc:.4f}")
print(f"Embedding Model Test Macro F1: {embed_test_macro_f1:.4f}\n")
print(classification_report(y_test_sent, embed_preds, target_names=['Negative', 'Positive']))

# Plot comparison
comparison_df = pd.DataFrame({
    'Model': ['TF-IDF + LR', 'Keras Learned Embeddings'],
    'Test Accuracy': [lr_test_acc, embed_test_acc],
    'Test Macro F1': [lr_test_macro_f1, embed_test_macro_f1]
})
print(comparison_df.to_string(index=False))

3064/3064 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step
Embedding Model Test Accuracy: 0.9336
Embedding Model Test Macro F1: 0.9044

              precision    recall  f1-score   support

    Negative       0.86      0.84      0.85     22163
    Positive       0.95      0.96      0.96     75855

    accuracy                           0.93     98018
   macro avg       0.91      0.90      0.90     98018
weighted avg       0.93      0.93      0.93     98018

                   Model  Test Accuracy  Test Macro F1
             TF-IDF + LR       0.886388       0.852256
Keras Learned Embeddings       0.933635       0.904379


## Part B: Semantic Product Search

Now, we build a search engine. We represent each product description as a dense embedding by averaging the embeddings of its constituent words (using the Keras Embedding weights trained in Part A).

We will then use **cosine similarity** to compare a user search query against all product descriptions and retrieve the top matches.

In [ ]:
# Extract the learned embedding matrix weights from the Keras model
embedding_layer = embed_model.layers[1] # input -> vectorizer -> embedding
weights = embedding_layer.get_weights()[0]
print(f"Embedding layer weights shape: {weights.shape} (vocab_size, embedding_dim)")

# Extract word index mapping from vectorize_layer
vocabulary = vectorize_layer.get_vocabulary()
word_to_index = {word: i for i, word in enumerate(vocabulary)}

def get_sentence_embedding(text):
    if not isinstance(text, str) or text.strip() == "":
        return np.zeros(EMBEDDING_DIM)

    # Tokenize words
    words = text.lower().split()
    indices = [word_to_index[w] for w in words if w in word_to_index and word_to_index[w] < VOCAB_SIZE]

    if len(indices) == 0:
        return np.zeros(EMBEDDING_DIM)

    # Extract weights for the words and calculate mean
    word_embeds = weights[indices]
    return np.mean(word_embeds, axis=0)

# Build unique product collection from meta train and test splits
prod_col = 'parent_asin' if 'parent_asin' in meta_train.columns else 'asin'
product_db = pd.concat([meta_train, meta_val, meta_test]).drop_duplicates(subset=[prod_col]).copy()
product_db = product_db[product_db['description_cleaned'] != ""].reset_index(drop=True)

print(f"Generating embeddings for {len(product_db)} product descriptions...")
desc_embeddings = np.array([get_sentence_embedding(desc) for desc in product_db['description_cleaned']])
print(f"Database embedding matrix shape: {desc_embeddings.shape}")

Embedding layer weights shape: (10000, 64) (vocab_size, embedding_dim)
Generating embeddings for 19160 product descriptions...
Database embedding matrix shape: (19160, 64)


### 1. Implement Search Query Interface
We implement a search function that computes the cosine similarity between the query embedding and the product database embeddings matrix.

In [ ]:
def semantic_search(query, top_n=5):
    query_vector = get_sentence_embedding(query).reshape(1, -1)

    # Calculate cosine similarities
    similarities = cosine_similarity(query_vector, desc_embeddings).flatten()

    # Sort indices in descending order
    top_indices = np.argsort(similarities)[::-1][:top_n]

    results = []
    for idx in top_indices:
        results.append({
            'score': similarities[idx],
            'title': product_db.iloc[idx]['title'],
            'store': product_db.iloc[idx]['store'],
            'price': product_db.iloc[idx]['price'],
            'description': product_db.iloc[idx]['description_cleaned'][:150] + "..."
        })
    return pd.DataFrame(results)

# Let's run a test queries
query1 = "organic hair growth moisturizing shampoo with coconut oil"
print(f"Search results for: '{query1}'")
semantic_search(query1)

Search results for: 'organic hair growth moisturizing shampoo with coconut oil'


,score,title,store,price,description
0,0.990700,(OGX) Organix Moroccan Surf Paste 4oz by (OGX)...,OGX,None,"Creates radical, unstructured, surf style, wit..."
1,0.989655,One Pack of 7 Horipenis Professional Tattoo St...,Horipenis,None,Enjoy crisp freehand stenciling with these pro...
2,0.989210,Botanical Beauty VITAMIN C Moisturizing Face O...,Botanical Beauty,12.95,Unique VITAMIN C Face Oil is an Anti- Aging ...
3,0.988828,"Earth Therapeutics ""Sole Food"" Foot Therapy Kit",Earth Therapeutics,None,"With active ingredients tea tree oil, chamomil..."
4,0.988671,Alpha-H Age Delay Hand & Cuticle Care Cream 100ml,Alpha-H,None,"A beautiful, rich, silky cream, designed to re..."


In [ ]:
query2 = "anti-aging facial serum cream for wrinkles"
print(f"Search results for: '{query2}'")
semantic_search(query2)

Search results for: 'anti-aging facial serum cream for wrinkles'


,score,title,store,price,description
0,0.986640,"Tru Beauty Electric Cleansing Facial Brush, Cl...",QQcute,None,"Beauty Electric Cleansing Facial Brush, Clean ..."
1,0.986507,The Gap Scents Heaven Eau De Toilette Perfume ...,GAP,None,Product Description Gap Scents Heaven Eau De T...
2,0.986507,The Gap Scents Heaven Eau De Toilette Perfume ...,GAP,None,Product Description Gap Scents Heaven Eau De T...
3,0.986331,Bath & Body Works - Gingham Love - Bundle -3 i...,Bath & Body Works,33.99,"3 items - Moisturizing Body Wash, Ultimate Hyd..."
4,0.986202,Victoria's Secret Mist & Lotion Gift Set Combo...,Victoria's Secret,41.96,Victoria's Secret XO Victorias Mist and Body L...
